In [9]:
from src.flow_match.flowMatch import FlowMatch
from src.flow_match.unet import UNet
from src.data.mnist import get_mnist_loader_and_transform
from src.data.cifar10 import get_cifar10_loader_and_transform
from torchvision.utils import save_image, make_grid
from src.flow_match.train import train
import matplotlib.pyplot as plt
import torch
import os

In [10]:
T = 1000
dataset = "cifar10" # can be "cifar10" or "mnist"

PATH_TO_READY_MODEL = None # input path for ready model

PATH_TO_SAVE_MODEL = f"flow_model_{dataset}.pth"
EPOCHS = 100 # for cifar10 it should be more than 1000, but for mnist 20-100 should be okay

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [12]:
if dataset == "mnist":
    data = get_mnist_loader_and_transform(batch_size=32)
elif dataset == "cifar10":
    data = get_cifar10_loader_and_transform(batch_size=32)

In [13]:
flow = FlowMatch(
    T = T,
    p_cond=0.2,
    time_shift=0.9,
    eps_model=UNet(
        in_channels=data.in_channels,
        out_channels=data.out_channels,
        T=T+1,
        num_classes=data.num_classes,
        steps=data.recommended_steps,
        attn_step=data.recommended_attn_step_indexes
    ),
    device=device
)
flow.to(device)
print(flow.device)


cuda


In [15]:
if PATH_TO_READY_MODEL is not None:
    flow.load_state_dict(torch.load(PATH_TO_READY_MODEL, map_location=device))
else:
    if os.path.exists(PATH_TO_SAVE_MODEL):
        flow.load_state_dict(torch.load(PATH_TO_SAVE_MODEL, map_location=device))
    _, val_losses = train(
        model=flow,
        optimizer=torch.optim.Adam(params=flow.parameters(), lr=2e-4),
        epochs=EPOCHS,
        device=device,
        train_loader=data.train_loader,
        val_loader=data.val_loader,
        save_path=PATH_TO_SAVE_MODEL,
        save_interval=100
    )

    path = PATH_TO_SAVE_MODEL if PATH_TO_SAVE_MODEL is not None else f"flow_model_{dataset}.pth"

    torch.save(flow.state_dict(), path)

    plt.plot(val_losses, label="Validation Loss")

    plt.legend()

loss for epoch 0: 0.2229:   0%|          | 0/1563 [08:06<?, ?it/s]

loss for epoch 0: 0.2229:   0%|          | 0/1563 [08:06<?, ?it/s]


loss for epoch 1: 0.1925:   0%|          | 0/1563 [00:07<?, ?it/s]

loss for epoch 1: 0.1944:   0%|          | 0/1563 [00:08<?, ?it/s]

loss for epoch 1: 0.2094:   0%|          | 0/1563 [00:08<?, ?it/s]

loss for epoch 1: 0.2077:   0%|          | 0/1563 [00:08<?, ?it/s]

loss for epoch 1: 0.2086:   0%|          | 0/1563 [00:09<?, ?it/s]

loss for epoch 1: 0.2060:   0%|          | 0/1563 [00:09<?, ?it/s]

loss for epoch 1: 0.2067:   0%|          | 0/1563 [00:09<?, ?it/s]

loss for epoch 1: 0.2054:   0%|          | 0/1563 [00:10<?, ?it/s]

loss for epoch 1: 0.2029:   0%|          | 0/1563 [00:10<?, ?it/s]

loss for epoch 1: 0.2055:   0%|          | 0/1563 [00:10<?, ?it/s]

loss for epoch 1: 0.2035:   0%|          | 0/1563 [00:10<?, ?it/s]

loss for epoch 1: 0.2046:   0%|          | 0/1563 [00:11<?, ?it/s]

loss for epoch 1: 0.2067:   0%|          | 0/15

KeyboardInterrupt: 

In [ ]:
n_samples = 10
classes = torch.full((n_samples,), 2, dtype=torch.long)
x_t = flow.sample(n_samples=n_samples, size=data.train_dataset[0][0].shape,classes=classes,cond_weight=0.8)

result = []
for i in range(x_t.shape[0]):
    result.append(data.transform_to_pil(x_t[i]))

grid = make_grid(x_t, nrow=10)
save_image(grid, f"sample.png")

cols = 5
rows = (n_samples // cols) + (0 if n_samples % cols == 0 else 1)
fig, axs = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
for i in range(len(result)):
    row = i // cols
    axs[row, i % cols].imshow(result[i], cmap='gray')